Here is the agenda, and every item on it traces back to something you saw in stage 2:

The 12 all-null rows. The mislabelled 2025-Q2 file — and separately, its 307 references that don’t overlap. The 44,842 duplicates. The 16 region spellings for 13 regions. Splitting city_district on / and stripping whitespace, which affects 835,467 rows. The area_m2 == 0 row. The fact that n_properties > 1 on some rows, meaning price_sar is a deal price, not a property price. The extreme tail — a 31.8 billion riyal price and a 24 million m² area. price_per_m2, which is both stored as text and is the leak. deal_ref, which is an ID and must never become a feature. And the five columns that are over 89% empty.

In [1]:
import pandas as pd, numpy as np, re

df = pd.read_parquet("../out/ksa_sales_raw.parquet")
before = len(df)

decisions = []   # (issue, decision, reasoning, rows_affected)
print("start:", before)

start: 1269512


In [2]:
df = df.dropna(subset=["price_sar", "date_greg"])

decisions.append((
    "12 rows null in every original column",
    "dropped",
    "their only non-null cells are quarter and source_file, both added by the "
    "loader; one per file in 12 of 23 files = trailing blank lines in the CSVs",
    before - len(df),
))
print("after blanks:", len(df))

after blanks: 1269500


In [3]:
# trust the data, not the filename
df["quarter"] = df.date_greg.dt.to_period("Q").astype(str)

decisions.append((
    "MOJ-Sales-2025-Q2.csv contains Apr-Jun 2024 data",
    "derive quarter from date_greg instead of the filename",
    "generic fix rather than deleting one file; a mislabelled filename can no "
    "longer corrupt a time-based split. 2025-Q2 is genuinely absent from the data",
    47518,
))

n1 = len(df)
df = df.drop_duplicates(subset="deal_ref", keep="first")

decisions.append((
    "47,211 transaction references appear twice",
    "drop_duplicates on deal_ref, keep first",
    "every repeated reference in the dataset comes from the republished file; "
    "deduplicating on the reference keeps the 307 genuinely new rows that "
    "deleting the whole file would have lost",
    n1 - len(df),
))
print("after dedup:", len(df))

after dedup: 1222289


In [4]:
pd.DataFrame(decisions, columns=["issue", "decision", "reasoning", "rows_affected"])

,issue,decision,reasoning,rows_affected
0,12 rows null in every original column,dropped,their only non-null cells are quarter and sour...,12
1,MOJ-Sales-2025-Q2.csv contains Apr-Jun 2024 data,derive quarter from date_greg instead of the f...,generic fix rather than deleting one file; a m...,47518
2,"47,211 transaction references appear twice","drop_duplicates on deal_ref, keep first",every repeated reference in the dataset comes ...,47211


In [5]:
# --- 4. location: region, city, district -----------------------------

def clean_ar(s):
    """Collapse whitespace; normalise word-final ه to ة."""
    if pd.isna(s):
        return s
    s = re.sub(r"\s+", " ", str(s).strip())
    return " ".join(re.sub(r"ه$", "ة", w) for w in s.split())

def clean_region(s):
    if pd.isna(s):
        return s
    s = re.sub(r"\s+", " ", str(s).strip())
    return clean_ar(re.sub(r"^(المنطقة|منطقة)\s+", "", s))

n_before = df.region.nunique()
df["region"] = df.region.map(clean_region)
df["city"]   = df.city.map(clean_ar)

# district: use the real column where it exists (2023-Q1 only),
# otherwise take the second half of city_district
split_district = df.city_district.str.split("/", n=1).str[1]
df["district"] = df.district.fillna(split_district).map(clean_ar)
df = df.drop(columns=["city_district"])

decisions.append((
    f"{n_before} region spellings for 13 real regions",
    "strip the leading منطقة/المنطقة, collapse whitespace, normalise word-final ه to ة",
    "without this, Makkah and the Eastern Province each split across two rows "
    "in every regional summary",
    int(n_before - df.region.nunique()),
))

print("regions :", df.region.nunique())
print("cities  :", df.city.nunique())
print("districts:", df.district.nunique(), "| nulls:", df.district.isna().sum())

regions : 13
cities  : 174
districts: 13781 | nulls: 0


In [6]:
df.district.value_counts().head(10)

district
أخرى          123576
الخير          25452
عريض           17566
نمار           17239
الجنادرية      14723
الرمال         13987
النظيم         13893
الريان         12781
العزيزية       11011
شرق الرياض     10730
Name: count, dtype: int64

In [7]:
df.district.str.startswith("حي ").sum()

np.int64(4294)

In [8]:
# --- 5. district: prefix, placeholder, and the city+district key -----

n_d = df.district.nunique()
df["district"] = df.district.str.replace(r"^حي\s+", "", regex=True)

decisions.append((
    "163 districts written both with and without the حي prefix",
    "strip a leading 'حي ' from district names",
    "4,294 rows affected; 'حي البوادي' and 'البوادي' are one district",
    int(n_d - df.district.nunique()),
))

df["location"] = df.city + "/" + df.district

decisions.append((
    "1,241 district names are reused across different cities",
    "build a combined city/district key and model on that, not on district alone",
    "770,395 rows (63%) carry an ambiguous district name — العزيزية in Riyadh "
    "and العزيزية in Jeddah are different places with different prices",
    770395,
))

decisions.append((
    "أخرى ('other') is 10.1% of rows, spread across 165 cities",
    "kept, but only inside the city/district key — never as a standalone district",
    "it is a placeholder for an unrecorded district, not a neighbourhood. "
    "Inside the key it still carries the city, so 'الرياض/أخرى' means "
    "'unspecified district in Riyadh', which is real information",
    123576,
))

print("districts:", df.district.nunique())
print("locations:", df.location.nunique())

districts: 13618
locations: 17310


In [9]:
vc = df.location.value_counts()
vc.describe()

count    17310.000000
mean        70.611727
std        449.088812
min          1.000000
25%          1.000000
50%          3.000000
75%         17.000000
max      25452.000000
Name: count, dtype: float64

In [10]:
(vc < 10).sum()

np.int64(11667)

In [11]:
# --- 6. errors fixed; scope decisions deferred ----------------------

n_z = int((df.area_m2 <= 0).sum())
df = df[df.area_m2 > 0]
decisions.append((
    "1 row with area_m2 = 0",
    "dropped",
    "a 1.36m SAR land plot with the area unrecorded; zero is not a measurement, "
    "and it makes price-per-m2 infinite, which propagates into every mean and quantile",
    n_z,
))

df = df.drop(columns=["price_per_m2", "plan", "parcel_no"])
decisions.append((
    "price_per_m2 is price_sar / area_m2, stored as text, 96% null",
    "column dropped",
    "it is a direct function of the target — the planted leak. plan and parcel_no "
    "dropped too: 96% null and not usable as features",
    0,
))

df["is_multi_property"] = df.n_properties > 1
decisions.append((
    "n_properties > 1 on 12,745 rows (1.04%)",
    "flagged, not dropped — excluded later at the scoping step",
    "price_sar is a DEAL price on those rows, so price-per-property is wrong. "
    "Flagging keeps the data available; dropping here would hide the decision",
    12745,
))

decisions.append((
    "extreme tails: price to 31.8bn, area to 24m m2",
    "kept — not errors",
    "1,558 of the 1,693 rows above 1m m2 are agricultural land, and 447 of the 635 "
    "above 100m SAR are commercial. A different market, not bad data. Scope is a "
    "modelling decision, made in the model notebook where it can be changed",
    0,
))

print("rows:", len(df), "| columns:", len(df.columns))

rows: 1222288 | columns: 15


In [12]:
log = pd.DataFrame(decisions, columns=["issue", "decision", "reasoning", "rows_affected"])
log

,issue,decision,reasoning,rows_affected
0,12 rows null in every original column,dropped,their only non-null cells are quarter and sour...,12
1,MOJ-Sales-2025-Q2.csv contains Apr-Jun 2024 data,derive quarter from date_greg instead of the f...,generic fix rather than deleting one file; a m...,47518
2,"47,211 transaction references appear twice","drop_duplicates on deal_ref, keep first",every repeated reference in the dataset comes ...,47211
3,16 region spellings for 13 real regions,"strip the leading منطقة/المنطقة, collapse whit...","without this, Makkah and the Eastern Province ...",3
4,163 districts written both with and without th...,strip a leading 'حي ' from district names,"4,294 rows affected; 'حي البوادي' and 'البوادي...",163
5,"1,241 district names are reused across differe...",build a combined city/district key and model o...,"770,395 rows (63%) carry an ambiguous district...",770395
6,"أخرى ('other') is 10.1% of rows, spread across...","kept, but only inside the city/district key — ...",it is a placeholder for an unrecorded district...,123576
7,1 row with area_m2 = 0,dropped,a 1.36m SAR land plot with the area unrecorded...,1
8,"price_per_m2 is price_sar / area_m2, stored as...",column dropped,it is a direct function of the target — the pl...,0
9,"n_properties > 1 on 12,745 rows (1.04%)","flagged, not dropped — excluded later at the s...","price_sar is a DEAL price on those rows, so pr...",12745


In [13]:
from pathlib import Path

df.to_parquet("../out/ksa_sales_clean.parquet", index=False)
log.to_csv("../out/cleaning_decisions.csv", index=False)

print(f"{before:,} rows in  ->  {len(df):,} rows out  "
      f"({1 - len(df)/before:.1%} removed)")
print("columns:", list(df.columns))

1,269,512 rows in  ->  1,222,288 rows out  (3.7% removed)
columns: ['quarter', 'date_greg', 'date_hijri', 'region', 'city', 'district', 'classification', 'property_type', 'n_properties', 'area_m2', 'price_sar', 'deal_ref', 'source_file', 'location', 'is_multi_property']
